In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.discriminant_analysis import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC
from sklearn.feature_selection import VarianceThreshold
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import f1_score



## Ler DATASET XGBOOST

In [ ]:
data_xgoost = pd.read_csv('/home/cid34senhas/Desktop/DAATP/Entrega/2_data_XGBoost.csv')

#### Fun??o Auxiliar

In [ ]:
def plot_confusion_matrix_with_labels(confusion_matrix, label_mapping):
    # Criar um mapeamento inverso
    reverse_label_mapping = {v: k for k, v in label_mapping.items()}
    
    # Aplicar o mapeamento inverso na matriz de confus?o
    cm_with_labels = np.zeros_like(confusion_matrix, dtype=object)
    for i in range(confusion_matrix.shape[0]):
        for j in range(confusion_matrix.shape[1]):
            cm_with_labels[i, j] = f"{reverse_label_mapping[i]} (Pred: {reverse_label_mapping[j]})"

    # Criar um DataFrame para facilitar a visualiza??o
    df_cm = pd.DataFrame(confusion_matrix, index=reverse_label_mapping.values(), columns=reverse_label_mapping.values())
    
    # Plotar a matriz de confus?o
    plt.figure(figsize=(10, 7))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix with Labels')
    plt.show()
label_mapping = {
    'CN-CN': 0,
    'AD-AD': 1,
    'MCI-AD': 2,
    'MCI-MCI': 3,
    'CN-MCI' : 4
}

# Modelo

In [ ]:
X = data_xgoost.drop(['Transition'], axis= 1)
y = data_xgoost['Transition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=34, stratify=y)


In [ ]:
xgboost_model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05,colsample_bytree=0.4, subsample=0.8, random_state=34)

xgboost_model.fit(X_train, y_train)
xgboost_pred = xgboost_model.predict(X_test)
print(classification_report(y_test, xgboost_pred,zero_division=0))
cm = confusion_matrix(y_test, xgboost_pred)
plot_confusion_matrix_with_labels(cm,label_mapping)

In [ ]:
xgboost_score = cross_val_score(xgboost_model, X, y, cv=5,scoring='f1_macro')

print(f"Cross Validation Score: {xgboost_score}")
print(f"Cross Validation Score M?dia: {xgboost_score.mean()}")
print(f"Cross Validation Score STD: {xgboost_score.std()}")

## Normaliza??o

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_norm = data_xgoost.drop(['Transition'], axis= 1)
y_norm = data_xgoost['Transition']
X_norm = scaler.fit_transform(X_train_norm)

In [ ]:
X_train_norm, X_test, y_train_norm, y_test = train_test_split(X_norm, y_norm, test_size=0.25, random_state=34, stratify=y)


In [ ]:
xgboost_model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05,colsample_bytree=0.4, subsample=0.8, random_state=34)

xgboost_model.fit(X_train_norm, y_train_norm)
xgboost_pred = xgboost_model.predict(X_test)
print(classification_report(y_test, xgboost_pred,zero_division=0))
cm = confusion_matrix(y_test, xgboost_pred)
plot_confusion_matrix_with_labels(cm,label_mapping)

In [ ]:
xgboost_score = cross_val_score(xgboost_model, X_norm, y_norm, cv=5,scoring='f1_macro')

print(f"Cross Validation Score: {xgboost_score}")
print(f"Cross Validation Score M?dia: {xgboost_score.mean()}")
print(f"Cross Validation Score STD: {xgboost_score.std()}")

Podemos perceber, pelo valor do cross Validation que o xgboost benificiou positivamente de forma ligueira da normaliza??o dos dados. 

## SMOTE

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_norm, y_norm, test_size=0.25, random_state=34, stratify=y)


print("Y Train Values:")
print(y_train.value_counts())


smote = SMOTE(sampling_strategy="minority", random_state=34)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


print("Y Train Values:")
print(y_train_smote.value_counts())

xg_modelo_smote =XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05,colsample_bytree=0.4, subsample=0.8, random_state=34)

xg_modelo_smote.fit(X_train_smote, y_train_smote)
xg_smote_pred = xg_modelo_smote.predict(X_test)
print(classification_report(y_test, xg_smote_pred,zero_division=0))
cm = confusion_matrix(y_test, xg_smote_pred)
plot_confusion_matrix_with_labels(cm,label_mapping)



Com o smote percebemos que a metrica de ***sampling_strategy="minority"*** havia uma diminui??o ligueira na fold criada no train_test_split, decidimos portanto n?o implementar o smote na nossa solu??o

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def feature_importance(model, X_train):

    # Obter as import?ncias das features
    importances = model.feature_importances_

    # Exibir os intervalos de import?ncia
    thresholds = [(0.012, float('inf')), (0.010, 0.012), (0.008, 0.010), 
                  (0.006, 0.008), (0.004, 0.006), (0.002, 0.004), (0, 0.002)]
    
    print("\nN?mero de features por intervalo de import?ncia:")
    for low, high in thresholds:
        count = np.sum((importances > low) & (importances <= high))
        print(f"Entre {low:.3f} e {high:.3f}: {count}")

    # Ordenar as import?ncias em ordem decrescente
    indices = np.argsort(importances)[::-1]

    # Criar o gr?fico
    plt.figure(figsize=(10, 6))
    plt.title("Feature Importance")
    
    # Adicionar barras
    plt.bar(range(X_train.shape[1]), importances[indices])

    # Exibir o gr?fico
    plt.tight_layout()
    plt.show()



feature_importance(xgboost_model,X_train)

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
import numpy as np


def evaluate_model_with_thresholds(model, X_train, X_test, y_train, y_test, thresholds, label_mapping,X,y):
    """
    Avalia o modelo RandomForestClassifier alterando o threshold para sele??o de features.

    Par?metros:
        model: O modelo RandomForestClassifier a ser utilizado.
        X_train: Conjunto de treinamento.
        X_test: Conjunto de teste.
        y_train: Labels do conjunto de treinamento.
        y_test: Labels do conjunto de teste.
        thresholds: Lista de thresholds para sele??o de features.
        label_mapping: Mapeamento de r?tulos para a matriz de confus?o.
    """
    for threshold in thresholds:
        print(f"\n--- Avaliando com threshold: {threshold} ---")

        # Criar objeto SelectFromModel com o threshold atual
        sfm = SelectFromModel(model, threshold=threshold)

        # Selecionar as features mais importantes
        X_important_train = sfm.fit_transform(X_train, y_train)
        X_important_test = sfm.transform(X_test)

        # Treinar o modelo com as features selecionadas
        model.fit(X_important_train, y_train)
        
        # Fazer previs?es
        y_pred = model.predict(X_important_test)

        # Imprimir relat?rio de classifica??o
        print(classification_report(y_test, y_pred, zero_division=0))

        # Calcular e plotar matriz de confus?o
        cm = confusion_matrix(y_test, y_pred)
        plot_confusion_matrix_with_labels(cm, label_mapping)

        # Cross-validation no conjunto original
        score_norm = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro')

        # Exibir resultados de valida??o cruzada
        print(f"Cross Validation Score: {score_norm}")
        print(f"Cross Validation Score M?dia: {score_norm.mean()}")
        print(f"Cross Validation Score STD: {score_norm.std()}")

# Exemplo de uso
thresholds = [0.012, 0.010, 0.008, 0.006, 0.002]

# Substitua pelas suas vari?veis reais de X_train, X_test, y_train, y_test e label_mapping
evaluate_model_with_thresholds(xgboost_model, X_train_norm, X_test, y_train_norm, y_test, thresholds, label_mapping,X_norm,y)
